# Example 4 — 2D Allen–Cahn Bimodal Posterior

Reproduces **Figures 5–8** of Alberts & Bilionis (2023). 2D Allen–Cahn energy U_ε[φ] on [−1, 1]² with ε = 0.01, observed on three of four boundaries (15 points each, σ² = 0.01²). Double-well structure produces a bimodal posterior; modes separated by a Gaussian mixture; per-mode predictions visualized.

In [ ]:
# Colab bootstrap — installs CUDA-enabled JAX. No-op locally.
import os, sys
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    %cd /content
    !nvidia-smi -L || echo 'No GPU detected — Runtime > Change runtime type > GPU'
    !rm -rf /content/pift-od-il-inverse-problems
    !git clone https://github.com/cmhobbs96/pift-od-il-inverse-problems.git /content/pift-od-il-inverse-problems
    assert Path('/content/pift-od-il-inverse-problems/pyproject.toml').exists()
    %cd /content/pift-od-il-inverse-problems
    !pip install -q --upgrade pip
    !pip install -q -e .
    !pip install -q --upgrade "jax[cuda12]"
    os.environ['PIFT_FORCE_BACKEND'] = 'local'
    os.environ['PIFT_FORCE_DEVICE'] = 'gpu'
    os.environ['JAX_PLATFORMS'] = 'cuda'
    os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
    print('Colab bootstrap complete.')
else:
    print('Not on Colab — bootstrap skipped.')

In [ ]:
import jax
print('JAX backend :', jax.default_backend())
print('JAX devices :', jax.devices())
if ON_COLAB:
    assert any(d.platform == 'gpu' for d in jax.devices()), 'CUDA GPU not visible to JAX'
    print('CUDA enabled ✓')

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
DEVICE = 'gpu' if ON_COLAB else 'cpu'
print('Repo root:', ROOT, '| device_preference:', DEVICE)

In [ ]:
from src.pipelines.phase_d_allen_cahn import run_phase_d_allen_cahn
result = run_phase_d_allen_cahn(device_preference=DEVICE)
print('status:', result['status'], '| runtime (s):', round(result.get('runtime_sec', 0), 1))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
for path in result.get('artifacts', []):
    if str(path).endswith('.png'):
        fig, ax = plt.subplots(figsize=(11, 9))
        ax.imshow(mpimg.imread(path)); ax.set_title(Path(path).name); ax.axis('off')
        plt.show()

**Expected:**
- **Fig. 5** prior samples illustrating bimodal Allen–Cahn structure.
- **Fig. 6** marginal posterior of basis coefficients showing two modes.
- **Fig. 7** ground truth, posterior median, and per-mode predicted fields.
- **Fig. 8** per-mode prediction error fields.